# Here

# **Stacking**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

## Loading Data

In [ ]:
url = "https://raw.githubusercontent.com/FatemaTaher/Optimized_Heart_disease_prediction/main/heart_disease_dir.csv"
df = pd.read_csv(url)

df = df.drop_duplicates()
#df['SkinCancer'] = df['SkinCancer'].fillna('No')
df['HeartDisease'] = df['HeartDisease'].map({'No': 0, 'Yes': 1})

## Feature Engineering

In [ ]:

df['Diabetic_Simple'] = (df['Diabetic'] != 'No').astype(int)

df['Risk_Score'] = (
    df['AgeCategory'].map({
        '18-24':1,'25-29':1,'30-34':2,'35-39':2,'40-44':3,
        '45-49':3,'50-54':4,'55-59':4,'60-64':5,'65-69':5,
        '70-74':6,'75-79':6,'80 or older':7
    }).fillna(3) * 2 +
    df['GenHealth'].map({'Poor':5,'Fair':4,'Good':3,'Very good':2,'Excellent':1}).fillna(3) * 1.5 +
    (df['Stroke'] == 'Yes').astype(int) * 5 +
    (df['DiffWalking'] == 'Yes').astype(int) * 3 +
    (df['Sex'] == 'Male').astype(int) * 2 +
    df['Diabetic_Simple'] * 2
)


## Classify Columns

In [ ]:
num_features = ['BMI', 'Risk_Score']
ordinal = ['AgeCategory', 'GenHealth']
binary = ['Sex', 'Stroke', 'DiffWalking', 'Diabetic_Simple']

age_order = ['18-24','25-29','30-34','35-39','40-44','45-49',
             '50-54','55-59','60-64','65-69','70-74','75-79','80 or older']
health_order = ['Poor','Fair','Good','Very good','Excellent']

X = df[num_features + ordinal + binary]
y = df['HeartDisease']

## Splitting


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution:\n{y.value_counts(normalize=True) * 100}")


## Preprocessing

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', RobustScaler())
    ]), num_features),
    ('ord', OrdinalEncoder(categories=[age_order, health_order]), ordinal),
    ('bin', OrdinalEncoder(), binary)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed shape: {X_train_processed.shape}")

## Model 1: Isolation Forest (Anomaly Detection)

In [ ]:
print("\n" + "="*50)
print("Training Isolation Forest...")
print("="*50)

X_train_normal = X_train_processed[y_train == 0]
print(f"Normal samples for training: {len(X_train_normal)}")

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.085,  # yes average
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_normal)

iso_pred = iso_forest.predict(X_test_processed)
iso_scores = -iso_forest.score_samples(X_test_processed)
iso_pred_binary = (iso_pred == -1).astype(int)

print("\n--- Isolation Forest Results ---")
print(classification_report(y_test, iso_pred_binary, target_names=['No', 'Yes']))
print(f"ROC-AUC: {roc_auc_score(y_test, iso_scores):.4f}")


## Model 2: XGBoost (Traditional)

In [ ]:

print("\n" + "="*50)
print("Training XGBoost...")
print("="*50)

smote = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print(f"After SMOTE: {X_train_smote.shape}")
print(f"Class distribution after SMOTE:\n{pd.Series(y_train_smote).value_counts()}")

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_smote, y_train_smote)
xgb_probs = xgb.predict_proba(X_test_processed)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, xgb_probs)

precisions = precisions[:-1]
recalls = recalls[:-1]

target_recall = 0.70
valid_idx = np.where(recalls >= target_recall)[0]

if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    best_thresh = thresholds[best_idx]
else:
    best_thresh = 0.3

xgb_pred = (xgb_probs >= best_thresh).astype(int)

print(f"\n--- XGBoost Results (threshold={best_thresh:.3f}) ---")
print(classification_report(y_test, xgb_pred, target_names=['No', 'Yes']))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")




## Ensemble(High Recall)

In [ ]:

print("\n" + "="*50)
print("Ensemble Results...")
print("="*50)

iso_scores_norm = iso_scores / np.max(iso_scores)

ensemble_pred = np.maximum(iso_pred_binary, xgb_pred)
ensemble_probs = np.maximum(iso_scores_norm, xgb_probs)

print("\n--- Ensemble (OR Logic) ---")
print(classification_report(y_test, ensemble_pred, target_names=['No', 'Yes']))
print(f"ROC-AUC: {roc_auc_score(y_test, ensemble_probs):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, ensemble_pred))

## Stacking

In [9]:

print("\n" + "="*50)
print("Training Stacking Meta-Learner...")
print("="*50)

meta_features_train = np.column_stack([
    -iso_forest.score_samples(X_train_processed) / np.max(-iso_forest.score_samples(X_train_processed)),
    xgb.predict_proba(X_train_processed)[:, 1]
])

meta_features_test = np.column_stack([
    iso_scores_norm,
    xgb_probs
])

meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000)
meta_learner.fit(meta_features_train, y_train)

stack_probs = meta_learner.predict_proba(meta_features_test)[:, 1]
stack_pred = (stack_probs >= 0.5).astype(int)

print("\n--- Stacking Results ---")
print(classification_report(y_test, stack_pred, target_names=['No', 'Yes']))
print(f"ROC-AUC: {roc_auc_score(y_test, stack_probs):.4f}")




Training: (239004, 8), Test: (59751, 8)
Class distribution:
HeartDisease
0    90.90693
1     9.09307
Name: proportion, dtype: float64
Processed shape: (239004, 8)

Training Isolation Forest...
Normal samples for training: 217271

--- Isolation Forest Results ---
              precision    recall  f1-score   support

          No       0.93      0.91      0.92     54318
         Yes       0.29      0.35      0.32      5433

    accuracy                           0.86     59751
   macro avg       0.61      0.63      0.62     59751
weighted avg       0.88      0.86      0.87     59751

ROC-AUC: 0.7132

Training XGBoost...
After SMOTE: (282452, 8)
Class distribution after SMOTE:
HeartDisease
0    217271
1     65181
Name: count, dtype: int64

--- XGBoost Results (threshold=0.690) ---
              precision    recall  f1-score   support

          No       0.96      0.79      0.87     54318
         Yes       0.25      0.70      0.37      5433

    accuracy                           0.78   

'joblib.dump({\n    \'preprocessor\': preprocessor,\n    \'iso_forest\': iso_forest,\n    \'xgb\': xgb,\n    \'meta_learner\': meta_learner,\n    \'threshold\': best_thresh\n}, \'anomaly_ensemble_model.pkl\')\n\nprint("\n✅ All models saved to \'anomaly_ensemble_model.pkl\'")'

## Summary

In [ ]:
## Summary

In [ ]:

print("\n" + "="*50)
print("FINAL COMPARISON")
print("="*50)

results = {
    'Isolation Forest': {'pred': iso_pred_binary, 'probs': iso_scores_norm},
    'XGBoost': {'pred': xgb_pred, 'probs': xgb_probs},
    'Ensemble (OR)': {'pred': ensemble_pred, 'probs': ensemble_probs},
    'Stacking': {'pred': stack_pred, 'probs': stack_probs}
}

for name, res in results.items():
    tn, fp, fn, tp = confusion_matrix(y_test, res['pred']).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    auc = roc_auc_score(y_test, res['probs'])

    print(f"\n{name}:")
    print(f"  Recall (Yes):    {recall:.3f}")
    print(f"  Precision (Yes): {precision:.3f}")
    print(f"  ROC-AUC:         {auc:.4f}")
    print(f"  TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")

# **Ideal**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

## Load & Clean

In [2]:
url = "https://raw.githubusercontent.com/FatemaTaher/Optimized_Heart_disease_prediction/main/heart_disease_dir.csv"
df = pd.read_csv(url)
df = df.drop_duplicates()
df['SkinCancer'] = df['SkinCancer'].fillna('No')
df['HeartDisease'] = df['HeartDisease'].map({'No': 0, 'Yes': 1})

## Features (optimized)

In [3]:
df['Diabetic_Simple'] = (df['Diabetic'] != 'No').astype(int)

df['Risk_Score'] = (
    df['AgeCategory'].map({
        '18-24':1,'25-29':1,'30-34':2,'35-39':2,'40-44':3,
        '45-49':3,'50-54':4,'55-59':4,'60-64':5,'65-69':5,
        '70-74':6,'75-79':6,'80 or older':7
    }).fillna(3) * 2 +
    df['GenHealth'].map({'Poor':5,'Fair':4,'Good':3,'Very good':2,'Excellent':1}).fillna(3) * 1.5 +
    (df['Stroke'] == 'Yes').astype(int) * 5 +
    (df['DiffWalking'] == 'Yes').astype(int) * 3 +
    (df['Sex'] == 'Male').astype(int) * 2 +
    df['Diabetic_Simple'] * 2
)


## Columns

In [4]:
num_features = ['BMI', 'Risk_Score']
ordinal = ['AgeCategory', 'GenHealth']
binary = ['Sex', 'Stroke', 'DiffWalking', 'Diabetic_Simple']

age_order = ['18-24','25-29','30-34','35-39','40-44','45-49',
             '50-54','55-59','60-64','65-69','70-74','75-79','80 or older']
health_order = ['Poor','Fair','Good','Very good','Excellent']

X = df[num_features + ordinal + binary]
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Preprocessing

In [5]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_features),
    ('ord', OrdinalEncoder(categories=[age_order, health_order]), ordinal),
    ('bin', OrdinalEncoder(), binary)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


## Model 1: Isolation Forest

In [6]:
X_train_normal = X_train_processed[y_train == 0]

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.085,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_normal)

iso_scores = -iso_forest.score_samples(X_test_processed)
iso_scores_train = -iso_forest.score_samples(X_train_processed)

## Model 2: XGBoost

In [7]:
smote = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train_smote, y_train_smote)

xgb_probs = xgb.predict_proba(X_test_processed)[:, 1]
xgb_probs_train = xgb.predict_proba(X_train_processed)[:, 1]

## Stacking (The Best )

In [8]:
iso_scores_norm = iso_scores / np.max(iso_scores)
iso_scores_train_norm = iso_scores_train / np.max(iso_scores_train)

meta_features_train = np.column_stack([iso_scores_train_norm, xgb_probs_train])
meta_features_test = np.column_stack([iso_scores_norm, xgb_probs])

meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000)
meta_learner.fit(meta_features_train, y_train)

stack_probs = meta_learner.predict_proba(meta_features_test)[:, 1]
stack_pred = (stack_probs >= 0.5).astype(int)

## Results

In [9]:

print("=== STACKING MODEL (Production) ===")
print(classification_report(y_test, stack_pred, target_names=['No', 'Yes']))
print(f"ROC-AUC: {roc_auc_score(y_test, stack_probs):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, stack_pred))



=== STACKING MODEL (Production) ===
              precision    recall  f1-score   support

          No       0.97      0.70      0.81     54318
         Yes       0.21      0.81      0.34      5433

    accuracy                           0.71     59751
   macro avg       0.59      0.75      0.57     59751
weighted avg       0.90      0.71      0.77     59751

ROC-AUC: 0.8289
Confusion Matrix:
[[37979 16339]
 [ 1038  4395]]
